In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base_path = '/content/drive/My Drive/MIMIC/'

df = pd.read_csv(base_path + 'training_dataset.csv')


from sklearn.model_selection import GroupKFold
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

import re  # Added regular expressions module
# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
                   'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
                   'vital_mean_spo2', 'vital_mean_temp']


from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# 5. Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install pytorch-tabular[extra]

In [4]:
import torch
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

# 1. Dynamically identify your categorical/one-hot binary columns
# (This captures all your 'Dx_...' columns and text/gender flags)
categorical_cols = [col for col in X_train.columns if col not in continuous_cols]

# 2. Recombine features and targets into single DataFrames for PyTorch Tabular
train_df = X_train.copy()
train_df['hospital_expire_flag'] = y_train.values

test_df = X_test.copy()
test_df['hospital_expire_flag'] = y_test.values

# 3. Configure the Data Settings
data_config = DataConfig(
    target=['hospital_expire_flag'],
    continuous_cols=continuous_cols,
    categorical_cols=categorical_cols,
)

# 4. Leverage the High-End GPU Server
# 4. Configured to run on standard CPU safely
trainer_config = TrainerConfig(
    batch_size=256,
    max_epochs=5,        # Temporary low epochs (5) so it completes quickly on CPU
    accelerator="cpu",   # Kept as CPU
    devices=1            # FIXED: Changed from "auto" string to integer 1
)

# 5. Define the Novel FT-Transformer Parameters
# 5. Optimized Light FT-Transformer Parameters to prevent RAM Crashes
model_config = FTTransformerConfig(
    task="classification",
    num_heads=2,          # Reduced from 4 to 2 to save attention matrix space
    num_attn_blocks=1,    # Reduced from 3 to 1 to drastically drop memory usage
    input_embed_dim=16,   # Explicitly limits the hidden token embedding size
    learning_rate=1e-3
)

optimizer_config = OptimizerConfig()

# 6. Initialize and Train the Model on the GPU
print("Initializing Proposed Tabular FT-Transformer Model on GPU...")
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

print("Training the Deep Learning Transformer...")
# tabular_model.fit(train_df=train_df)
tabular_model.fit(train=train_df)

# 7. Evaluate Performance on Holdout Test Set
print("\n================ EVALUATING TRANSFORMER MODEL ==================")
result = tabular_model.evaluate(test_df)
predictions = tabular_model.predict(test_df)

# Output final deep learning classification probabilities
y_prob_transformer = predictions['1_probability'].values
print("\nEvaluation metrics on holdout test set:")
print(result)

2026-06-05 10:01:12,774 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-06-05 10:01:12,843 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders


Initializing Proposed Tabular FT-Transformer Model on GPU...
Training the Deep Learning Transformer...


2026-06-05 10:01:12,990 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-05 10:01:15,345 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-06-05 10:01:17,541 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-05 10:01:17,685 - {pytorch_tabular.tabular_model:677} - INFO - Training Started


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  5.2 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  1.2 M │ train │     0 │
│ 2 │ _head            │ LinearHead            │     34 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss      │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 1.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.2 M                                                                                                
Total estimated model params size (MB): 4.846                                                                      
Modules in train mode: 1216                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...


================ EVALUATING TRANSFORMER MODEL ==================


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.9504364728927612     │
│         test_loss         │    0.12951979041099548    │
│        test_loss_0        │    0.12951979041099548    │
└───────────────────────────┴───────────────────────────┘

KeyError: '1_probability'

In [6]:
# 1. Let's see exactly what columns your predictions dataframe actually has
print("Available prediction columns:", predictions.columns.tolist())

# 2. Dynamic Safe Extraction: Find the correct probability column automatically
if '1_probability' in predictions.columns:
    y_prob_transformer = predictions['1_probability'].values
elif 'prediction' in predictions.columns:
    y_prob_transformer = predictions['prediction'].values
else:
    # If it outputted raw probabilities for both classes [prob_0, prob_1], take class 1
    prob_cols = [col for col in predictions.columns if 'prob' in col.lower() or 'class' in col.lower()]
    if len(prob_cols) >= 2:
        y_prob_transformer = predictions[prob_cols[1]].values
    else:
        # Fallback: find the float column that isn't the original flag
        y_prob_transformer = predictions.select_dtypes(include=[np.float32, np.float64]).iloc[:, -1].values

print("\nSuccessfully extracted probabilities! Sample:", y_prob_transformer[:5])

Available prediction columns: ['hospital_expire_flag_0_probability', 'hospital_expire_flag_1_probability', 'hospital_expire_flag_prediction']

Successfully extracted probabilities! Sample: [0.00050345 0.00049472 0.00048633 0.00048806 0.0004974 ]


In [8]:
# Reconstruct df_combined quickly from your loaded dataframes
# so the test_idx can find the subject_ids
try:
    # If you used df3 and df4 to build it originally:
    df4_aligned = df4[df3.columns]
    df_combined = pd.concat([df3, df4_aligned], axis=0, ignore_index=True)
except NameError:
    # Fallback: If your notebook just loaded 'df' from 'training_dataset.csv' at the top:
    df_combined = df.copy()

print("df_combined successfully restored! Shape:", df_combined.shape)

df_combined successfully restored! Shape: (17755, 93)


In [9]:
from sklearn.metrics import classification_report, roc_auc_score

# 1. Create the evaluation summary linking predictions back to the patient indices
test_summary_transformer = pd.DataFrame({
    'subject_id': df_combined.iloc[test_idx]['subject_id'].values,
    'y_true': y_test.values,
    'y_prob': y_prob_transformer
})

# 2. Group by subject_id to collapse down to patient-level metrics
patient_res_trans = test_summary_transformer.groupby('subject_id').mean()
# If any admission for a patient was a 1, their actual true target is 1
patient_true_labels = test_summary_transformer.groupby('subject_id')['y_true'].max()

# 3. Apply the classification cutoff threshold at 50%
preds_transformer = (patient_res_trans['y_prob'] >= 0.5).astype(int)

print("\n--- Proposed FT-Transformer Patient-Level Evaluation ---")
print(classification_report(patient_true_labels, preds_transformer))
print(f"Transformer Patient-Level ROC-AUC Score: {roc_auc_score(patient_true_labels, patient_res_trans['y_prob']):.4f}")


--- Proposed FT-Transformer Patient-Level Evaluation ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       628
           1       0.33      0.35      0.34        17

    accuracy                           0.96       645
   macro avg       0.66      0.67      0.66       645
weighted avg       0.97      0.96      0.96       645

Transformer Patient-Level ROC-AUC Score: 0.8995


In [10]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=0e80419b7e7e4e7d0aa330e92dfa6607dc0576d9a2b55105a1e39efc13c69d98
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Ensure LIME is available; install dynamically inline if missing
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    !pip install lime
    from lime.lime_tabular import LimeTabularExplainer

# ==============================================================================
# PREPARATION: Clean DataFrames for XAI Inputs
# ==============================================================================
X_train_clean = X_train.astype(float)
X_test_clean = X_test.astype(float)

# Select a single patient instance from the holdout split to inspect (Index 0)
patient_idx = 0
single_patient_df = X_test_clean.iloc[[patient_idx]]

# ==============================================================================
# 1. SHAP GLOBAL & LOCAL EXPLANATIONS (Using Native PyTorch Tabular Captum Engine)
# ==============================================================================
print("Extracting Neural Network SHAP attributions via PyTorch Tabular...")

# 'b|20' tells the model to automatically use 20 clustered rows as a fast baseline
shap_attributions_df = tabular_model.explain(
    data=X_test_clean,
    method="GradientShap",
    baselines="b|20"
)

# --- VISUALIZATION 1A: Global Summary Plot ---
print("Generating Global SHAP Summary Plot...")
plt.figure(figsize=(11, 6))
# Reconstruct a SHAP Explanation Object so it hooks into the default beeswarm visualizer
shap_exp = shap.Explanation(
    values=shap_attributions_df.values,
    base_values=np.zeros(len(X_test_clean)), # Centered normalization baseline
    data=X_test_clean.values,
    feature_names=X_test_clean.columns
)
shap.summary_plot(shap_exp, X_test_clean, max_display=15, show=False)
plt.title("Proposed Transformer Model - Global SHAP Feature Significance", fontsize=13, pad=20)
plt.tight_layout()
plt.show()

# --- VISUALIZATION 1B: Local Patient Waterfall Breakdown ---
print(f"Generating Local SHAP Waterfall Plot for Patient Index {patient_idx}...")
plt.figure(figsize=(12, 4))
shap.plots.waterfall(shap_exp[patient_idx], max_display=10, show=False)
plt.title(f"Transformer Local Risk Vectors (Test Row: {patient_idx})", fontsize=13, pad=20)
plt.tight_layout()
plt.show()


# ==============================================================================
# 2. LIME INTERPRETATION DASHBOARD
# ==============================================================================
print("\nInitializing LIME Tabular Explainer Framework...")

# Helper function to let LIME call our Tabular Transformer predictions smoothly
def transformer_predict_probabilities(X_numpy_array):
    df_temp = pd.DataFrame(X_numpy_array, columns=X_train_clean.columns)
    preds_df = tabular_model.predict(df_temp)
    # Reconstruct the 2D probability matrix [probability_survive, probability_expire]
    prob_1 = preds_df.select_dtypes(include=[np.float32, np.float64]).iloc[:, -1].values
    prob_0 = 1.0 - prob_1
    return np.vstack([prob_0, prob_1]).T

# Set up the tabular local surrogate builder
lime_explainer = LimeTabularExplainer(
    training_data=X_train_clean.values,
    feature_names=X_train_clean.columns,
    class_names=['Survives', 'Expires'],
    mode='classification',
    random_state=42
)

print(f"Executing local sample perturbations for Patient Index {patient_idx}...")
lime_interpretation = lime_explainer.explain_instance(
    data_row=X_test_clean.iloc[patient_idx],
    predict_fn=transformer_predict_probabilities,
    num_features=8
)

print("Displaying LIME Patient Risk Dashboard...")
lime_interpretation.show_in_notebook(show_table=True)

Extracting Neural Network SHAP attributions via PyTorch Tabular...


NameError: name 'tabular_model' is not defined

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Make sure LIME is available; install dynamically inline if missing
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    !pip install lime
    from lime.lime_tabular import LimeTabularExplainer

# ==============================================================================
# PREPARATION: Clean DataFrames for XAI Inputs
# ==============================================================================
X_train_clean = X_train.astype(float)
X_test_clean = X_test.astype(float)

# Select a single patient instance from the holdout split to inspect (Index 0)
patient_idx = 0
single_patient_df = X_test_clean.iloc[[patient_idx]]

# ==============================================================================
# 1. SHAP GLOBAL & LOCAL EXPLANATIONS (Using Native PyTorch Tabular Captum Engine)
# ==============================================================================
print("Extracting Neural Network SHAP attributions via PyTorch Tabular...")

# 'b|20' tells the model to automatically use 20 clustered rows as a fast baseline
shap_attributions_df = tabular_model.explain(
    data=X_test_clean,
    method="GradientShap",
    baselines="b|20"
)

# --- VISUALIZATION 1A: Global Summary Plot ---
print("Generating Global SHAP Summary Plot...")
plt.figure(figsize=(11, 6))
# Reconstruct a SHAP Explanation Object so it hooks into the default beeswarm visualizer
shap_exp = shap.Explanation(
    values=shap_attributions_df.values,
    base_values=np.zeros(len(X_test_clean)), # Centered normalization baseline
    data=X_test_clean.values,
    feature_names=X_test_clean.columns
)
shap.summary_plot(shap_exp, X_test_clean, max_display=15, show=False)
plt.title("Proposed Transformer Model - Global SHAP Feature Significance", fontsize=13, pad=20)
plt.tight_layout()
plt.show()

# --- VISUALIZATION 1B: Local Patient Waterfall Breakdown ---
print(f"Generating Local SHAP Waterfall Plot for Patient Index {patient_idx}...")
plt.figure(figsize=(12, 4))
shap.plots.waterfall(shap_exp[patient_idx], max_display=10, show=False)
plt.title(f"Transformer Local Risk Vectors (Test Row: {patient_idx})", fontsize=13, pad=20)
plt.tight_layout()
plt.show()


# ==============================================================================
# 2. LIME INTERPRETATION DASHBOARD
# ==============================================================================
print("\nInitializing LIME Tabular Explainer Framework...")

# Helper function to let LIME call our Tabular Transformer predictions smoothly
def transformer_predict_probabilities(X_numpy_array):
    df_temp = pd.DataFrame(X_numpy_array, columns=X_train_clean.columns)
    preds_df = tabular_model.predict(df_temp)
    # Reconstruct the 2D probability matrix [probability_survive, probability_expire]
    prob_1 = preds_df.select_dtypes(include=[np.float32, np.float64]).iloc[:, -1].values
    prob_0 = 1.0 - prob_1
    return np.vstack([prob_0, prob_1]).T

# Set up the tabular local surrogate builder
lime_explainer = LimeTabularExplainer(
    training_data=X_train_clean.values,
    feature_names=X_train_clean.columns,
    class_names=['Survives', 'Expires'],
    mode='classification',
    random_state=42
)

print(f"Executing local sample perturbations for Patient Index {patient_idx}...")
lime_interpretation = lime_explainer.explain_instance(
    data_row=X_test_clean.iloc[patient_idx],
    predict_fn=transformer_predict_probabilities,
    num_features=8
)

print("Displaying LIME Patient Risk Dashboard...")
lime_interpretation.show_in_notebook(show_table=True)

Extracting Neural Network SHAP attributions via PyTorch Tabular...


NameError: name 'tabular_model' is not defined